# Customer Churn Defuser — 01. Dataset Generation
Generates a realistic synthetic dataset of **10,000 customer records** with business-logic-driven churn labels (low login frequency, many support tickets, low engagement, many inactive days, and payment failures all raise churn probability).

Output: `dataset/customer_churn_dataset.csv`

## 1. Imports & Configuration

In [1]:
import numpy as np
import pandas as pd
import os

RANDOM_SEED = 42
NUM_RECORDS = 10000

np.random.seed(RANDOM_SEED)

## 2. Generate Demographic & Subscription Features

In [2]:
n = NUM_RECORDS

customer_id = np.arange(100000, 100000 + n)

age = np.random.randint(18, 70, size=n)
gender = np.random.choice(["Male", "Female", "Other"], size=n, p=[0.48, 0.48, 0.04])
subscription_type = np.random.choice(
    ["Basic", "Standard", "Premium"], size=n, p=[0.4, 0.4, 0.2]
)

base_fee = {"Basic": 9.99, "Standard": 19.99, "Premium": 39.99}
monthly_fee = np.array([base_fee[s] for s in subscription_type]) + np.round(
    np.random.normal(0, 2, size=n), 2
)
monthly_fee = np.clip(monthly_fee, 5, 100)

subscription_months = np.random.randint(1, 60, size=n)

## 3. Generate Behavioral & Engagement Features

In [3]:
login_frequency = np.random.poisson(lam=8, size=n)
login_frequency = np.clip(login_frequency, 0, 60)

session_duration = np.round(np.random.gamma(shape=2.0, scale=10.0, size=n), 2)
session_duration = np.clip(session_duration, 1, 180)

support_tickets = np.random.poisson(lam=1.2, size=n)
support_tickets = np.clip(support_tickets, 0, 15)

payment_failures = np.random.poisson(lam=0.4, size=n)
payment_failures = np.clip(payment_failures, 0, 10)

last_active_days = np.random.exponential(scale=15, size=n).astype(int)
last_active_days = np.clip(last_active_days, 0, 180)

customer_satisfaction = np.clip(
    np.round(np.random.normal(7, 1.8, size=n), 1), 1, 10
)

feature_usage_score = np.clip(
    np.round(np.random.normal(55, 20, size=n), 1), 0, 100
)

referrals = np.random.poisson(lam=0.6, size=n)
referrals = np.clip(referrals, 0, 10)

## 4. Compute Composite Engagement Score

In [4]:
engagement_score = np.clip(
    (
        0.3 * (login_frequency / 60 * 100)
        + 0.3 * feature_usage_score
        + 0.2 * (session_duration / 180 * 100)
        + 0.2 * (customer_satisfaction / 10 * 100)
    ),
    0,
    100,
)
engagement_score = np.round(engagement_score, 1)

## 5. Business-Logic Driven Churn Probability

Churn risk increases with: low login frequency, high support tickets, low engagement, high inactivity, and payment failures.

In [5]:
risk_score = (
    0.25 * (1 - login_frequency / max(login_frequency.max(), 1))
    + 0.20 * (support_tickets / max(support_tickets.max(), 1))
    + 0.20 * (1 - engagement_score / 100)
    + 0.20 * (last_active_days / max(last_active_days.max(), 1))
    + 0.15 * (payment_failures / max(payment_failures.max(), 1))
)

risk_score = (risk_score - risk_score.min()) / (risk_score.max() - risk_score.min())
churn_prob = np.clip(risk_score + np.random.normal(0, 0.07, size=n), 0, 1)

churn = (churn_prob > 0.55).astype(int)

## 6. Assemble & Inspect the DataFrame

In [6]:
df = pd.DataFrame(
    {
        "customer_id": customer_id,
        "age": age,
        "gender": gender,
        "subscription_type": subscription_type,
        "monthly_fee": monthly_fee,
        "subscription_months": subscription_months,
        "login_frequency": login_frequency,
        "session_duration": session_duration,
        "support_tickets": support_tickets,
        "payment_failures": payment_failures,
        "last_active_days": last_active_days,
        "customer_satisfaction": customer_satisfaction,
        "feature_usage_score": feature_usage_score,
        "referrals": referrals,
        "engagement_score": engagement_score,
        "churn": churn,
    }
)

df.head()

,customer_id,age,gender,subscription_type,monthly_fee,subscription_months,login_frequency,session_duration,support_tickets,payment_failures,last_active_days,customer_satisfaction,feature_usage_score,referrals,engagement_score,churn
0,100000,56,Female,Premium,40.11,30,4,18.21,0,1,18,8.7,82.4,2,46.1,1
1,100001,69,Male,Standard,19.87,39,10,14.30,1,1,13,8.2,76.0,1,45.8,0
2,100002,46,Male,Standard,22.88,15,6,8.15,1,0,16,8.3,84.8,1,45.9,0
3,100003,32,Female,Basic,13.12,46,2,4.04,0,0,22,8.4,70.4,1,39.4,0
4,100004,60,Male,Standard,19.71,58,7,20.25,1,1,6,2.7,64.7,0,30.6,1


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 16 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   customer_id            10000 non-null  int64  
 1   age                    10000 non-null  int64  
 2   gender                 10000 non-null  str    
 3   subscription_type      10000 non-null  str    
 4   monthly_fee            10000 non-null  float64
 5   subscription_months    10000 non-null  int64  
 6   login_frequency        10000 non-null  int64  
 7   session_duration       10000 non-null  float64
 8   support_tickets        10000 non-null  int64  
 9   payment_failures       10000 non-null  int64  
 10  last_active_days       10000 non-null  int64  
 11  customer_satisfaction  10000 non-null  float64
 12  feature_usage_score    10000 non-null  float64
 13  referrals              10000 non-null  int64  
 14  engagement_score       10000 non-null  float64
 15  churn         

In [8]:
print(f"Total records: {len(df)}")
print(f"Churn rate: {df['churn'].mean() * 100:.2f}%")

Total records: 10000
Churn rate: 24.01%


## 7. Save Dataset to CSV

In [9]:
os.makedirs("dataset", exist_ok=True)
out_path = os.path.join("dataset", "customer_churn_dataset.csv")
df.to_csv(out_path, index=False)
print(f"Dataset saved to: {out_path}")

Dataset saved to: dataset/customer_churn_dataset.csv
